# Этап 2. Очистка данных

Источник для очистки - Parquet с типизированными данными после первого этапа.

In [31]:
from pathlib import Path

import duckdb

cwd = Path.cwd()
BASE_DIR = cwd.parent if cwd.name == "notebooks" else cwd
parquet_path = BASE_DIR / "data/processed/raw_typed.parquet"

con = duckdb.connect()
con.read_parquet(str(parquet_path), file_row_number=True).create_view("cars_raw");

In [36]:
con.execute("PRAGMA disable_progress_bar");

## 2.1. Дубли объявлений

Для каждой ссылки остаётся запись с самой свежей датой сбора. При равных датах предпочтение у первой строки в Parquet. Строки без ссылки сохраняются.

In [32]:
con.execute("""
    CREATE OR REPLACE TEMP VIEW cars_dedup AS
    SELECT * EXCLUDE (file_row_number)
    FROM cars_raw
    QUALIFY NULLIF(TRIM(link), '') IS NULL
        OR ROW_NUMBER() OVER (
            PARTITION BY link
            ORDER BY parse_date DESC NULLS LAST, file_row_number
        ) = 1
""")

rows_before = con.execute("SELECT COUNT(*) FROM cars_raw").fetchone()[0]
rows_after = con.execute("SELECT COUNT(*) FROM cars_dedup").fetchone()[0]
print(f"Строк до очистки: {rows_before:,}\nСтрок после очистки: {rows_after:,}\nУдалено дублей: {rows_before - rows_after:,}")

Строк до очистки: 1,294,757
Строк после очистки: 950,457
Удалено дублей: 344,300


## 2.2. Выбросы цены

Для каждой пары марка-модель с минимум 100 положительными ценами задаются границы: 1-й и 99-й процентили. Цены на границах сохраняются. Порог в 100 записей, чтобы каждому проценту соответствовало хотя бы одно объявление. Для малых групп и строк без марки или модели сохраняются все положительные цены. Нулевые и отрицательные цены исключаются. Пропуски цены остаются до пункта 2.5.

In [33]:
con.execute("""
    CREATE OR REPLACE TEMP VIEW cars_price_clean AS

    WITH price_bounds AS (
        SELECT brand, name,
               quantile_disc(price, 0.01) AS lower_price,
               quantile_disc(price, 0.99) AS upper_price
        FROM cars_dedup
        WHERE price > 0 AND TRIM(brand) <> '' AND TRIM(name) <> ''
        GROUP BY brand, name
        HAVING COUNT(*) >= 100
    )
    
    SELECT cars.*
    FROM cars_dedup AS cars
    LEFT JOIN price_bounds AS bounds USING (brand, name)
    WHERE price IS NULL
       OR (price > 0 AND (lower_price IS NULL OR price BETWEEN lower_price AND upper_price))
""")

In [38]:
con.sql("SELECT * EXCLUDE (description) FROM cars_price_clean ORDER BY brand, name LIMIT 10").show()

┌─────────┬─────────┬────────────┬─────────────┬──────────┬───────┬─────────┬──────────────┬───────┬─────────┬──────────────────────────────────┬────────────┬────────────────────┬─────────────────────┬────────────────┬────────────────────────────────────────────────────────┬─────────────────────┐
│  brand  │  name   │  bodyType  │    color    │ fuelType │ year  │ mileage │ transmission │ power │  price  │       vehicleConfiguration       │ engineName │ engineDisplacement │        date         │    location    │                          link                          │     parse_date      │
│ varchar │ varchar │  varchar   │   varchar   │ varchar  │ int16 │  int32  │   varchar    │ int16 │  int64  │             varchar              │  varchar   │      varchar       │      timestamp      │    varchar     │                        varchar                         │      timestamp      │
├─────────┼─────────┼────────────┼─────────────┼──────────┼───────┼─────────┼──────────────┼───────┼──────

In [35]:
rows_after_price = con.execute("SELECT COUNT(*) FROM cars_price_clean").fetchone()[0]
print(f"Строк до фильтрации цен: {rows_after:,}")
print(f"Исключено по цене: {rows_after - rows_after_price:,}")
print(f"Строк после фильтрации: {rows_after_price:,}")

Строк до фильтрации цен: 950,457
Исключено по цене: 16,167
Строк после фильтрации: 934,290


## 2.3. Год, пробег и мощность

Допустимые значения (включительно): год выпуска от 1960 до 2023, мощность от 20 до 1000 л.с., пробег от 0 км.

Для объявлений, опубликованных в год выпуска, приведена сводка пробега. Автомобили с нулевым пробегом в более поздний год и объявления, опубликованные до года выпуска, учтены отдельно и сохранены для проверки.

In [48]:
con.execute("""
    CREATE OR REPLACE TEMP VIEW cars_specs_clean AS
    SELECT *
    FROM cars_price_clean
    WHERE (year IS NULL OR year BETWEEN 1960 AND 2023)
      AND (power IS NULL OR power BETWEEN 20 AND 1000)
      AND (mileage IS NULL OR mileage >= 0)
""")

rows_after_specs = con.execute("SELECT COUNT(*) FROM cars_specs_clean").fetchone()[0]
print(f"Исключено по году, пробегу или мощности: {rows_after_price - rows_after_specs:,}")
print(f"Строк после фильтрации: {rows_after_specs:,}")

Исключено по году, пробегу или мощности: 399
Строк после фильтрации: 933,891


In [ ]:
con.sql("""
    SELECT
        COUNT(*) AS "Объявления в год выпуска",
        COUNT(mileage) AS "С указанным пробегом",
        MEDIAN(mileage) AS "Медиана, км",
        MAX(mileage) AS "Максимум, км"
    FROM cars_specs_clean
    WHERE YEAR(date) = year
""").show()

┌──────────────────────────┬──────────────────────┬─────────────┬──────────────┐
│ Объявления в год выпуска │ С указанным пробегом │ Медиана, км │ Максимум, км │
│          int64           │        int64         │   double    │    int32     │
├──────────────────────────┼──────────────────────┼─────────────┼──────────────┤
│                     9270 │                   55 │      1000.0 │         5000 │
└──────────────────────────┴──────────────────────┴─────────────┴──────────────┘



## 2.4. Нормализация марок и моделей

Лишние пробелы удаляются. Для вариантов, различающихся только регистром, используется самое частое написание. Для моделей частота определяется внутри марки. При равной частоте выбирается первое написание в порядке сортировки.

In [52]:
con.execute(r"""
    CREATE OR REPLACE TEMP VIEW cars_categories_clean AS
    WITH trimmed AS (
        SELECT * REPLACE (
            trim(regexp_replace(brand, '\s+', ' ', 'g')) AS brand,
            trim(regexp_replace(name, '\s+', ' ', 'g')) AS name
        )
        FROM cars_specs_clean
    )
    SELECT * REPLACE (
        mode(brand ORDER BY brand) OVER (PARTITION BY lower(brand)) AS brand,
        mode(name ORDER BY name) OVER (PARTITION BY lower(brand), lower(name)) AS name
    )
    FROM trimmed
""")

In [54]:
con.sql("""SELECT * EXCLUDE (description) FROM cars_categories_clean LIMIT 10""").show()

┌────────────┬─────────┬───────────┬─────────────┬──────────┬───────┬─────────┬──────────────┬───────┬─────────┬─────────────────────────────┬────────────┬────────────────────┬─────────────────────┬─────────────────┬───────────────────────────────────────────────────────────────┬─────────────────────┐
│   brand    │  name   │ bodyType  │    color    │ fuelType │ year  │ mileage │ transmission │ power │  price  │    vehicleConfiguration     │ engineName │ engineDisplacement │        date         │    location     │                             link                              │     parse_date      │
│  varchar   │ varchar │  varchar  │   varchar   │ varchar  │ int16 │  int32  │   varchar    │ int16 │  int64  │           varchar           │  varchar   │      varchar       │      timestamp      │     varchar     │                            varchar                            │      timestamp      │
├────────────┼─────────┼───────────┼─────────────┼──────────┼───────┼─────────┼────────────